# GitHub Integration Test Suite (`test3.ipynb`)

This notebook tests the GitHub integration layer in `app/github/`:
1. **`diff_parser.py`**: Parse unified Git diffs into structured file changes and line numbers.
2. **`comment_poster.py`**: Post summary comments, inline review comments, and check runs via PyGithub.
3. **Mock & Live Testing**: Test error handling, validation, annotation batching, and optional live GitHub API calls.

In [1]:
# ── 1. Setup & Imports ──────────────────────────────────────────────────
import sys, os, json
from pathlib import Path
from unittest.mock import MagicMock

# Ensure agent-service directory is on sys.path
CWD = Path.cwd()
if str(CWD) not in sys.path:
    sys.path.insert(0, str(CWD))

from dotenv import load_dotenv
load_dotenv(CWD / ".env")

from app.github import (
    parse_diff,
    post_summary_comment,
    post_inline_comment,
    create_check_run,
    get_github_client,
    resolve_repository,
)

print("GitHub integration module loaded successfully!")
github_token = os.getenv("GITHUB_TOKEN")
print("GITHUB_TOKEN present in .env:", bool(github_token))
if github_token:
    print("Token preview :", github_token[:4] + "..." + github_token[-4:])

GitHub integration module loaded successfully!
GITHUB_TOKEN present in .env: False


### Test 1: `parse_diff` - Single File & Single Hunk
Parses a standard unified diff and verifies added lines (new line numbers) and removed lines (old line numbers).

In [2]:
sample_diff = """diff --git a/src/user.py b/src/user.py
--- a/src/user.py
+++ b/src/user.py
@@ -10,5 +10,7 @@
 def get_user(user_id):
-    query = "SELECT * FROM users"
+    query = f"SELECT * FROM users WHERE id = {user_id}"
+    result = db.execute(query)
     return result
"""

parsed = parse_diff(sample_diff)
print(f"Files parsed: {len(parsed)}")
print(json.dumps(parsed, indent=2))

assert len(parsed) == 1
entry = parsed[0]
assert entry["file"] == "src/user.py"
assert len(entry["added_lines"]) == 2
assert entry["added_lines"][0]["line_number"] == 11
assert entry["added_lines"][1]["line_number"] == 12
assert len(entry["removed_lines"]) == 1
assert entry["removed_lines"][0]["line_number"] == 11
print("\nTest 1 (Single Hunk): PASSED")

Files parsed: 1
[
  {
    "file": "src/user.py",
    "added_lines": [
      {
        "line_number": 11,
        "content": "    query = f\"SELECT * FROM users WHERE id = {user_id}\""
      },
      {
        "line_number": 12,
        "content": "    result = db.execute(query)"
      }
    ],
    "removed_lines": [
      {
        "line_number": 11,
        "content": "    query = \"SELECT * FROM users\""
      }
    ]
  }
]

Test 1 (Single Hunk): PASSED


### Test 2: `parse_diff` - Complex Diff (Multiple Files, New Files, Renames)
Verifies handling of newly created files, deleted files, and file renames across multiple files.

In [3]:
complex_diff = """
diff --git a/src/config.py b/src/config.py
--- a/src/config.py
+++ b/src/config.py
@@ -1,3 +1,4 @@
+DEBUG = True
 PORT = 8080
 
diff --git a/src/services/new_service.py b/src/services/new_service.py
new file mode 100644
--- /dev/null
+++ b/src/services/new_service.py
@@ -0,0 +1,3 @@
+class NewService:
+    def run(self):
+        pass
 
diff --git a/old_script.py b/new_script.py
similarity index 98%
rename from old_script.py
rename to new_script.py
--- a/old_script.py
+++ b/new_script.py
@@ -5,2 +5,2 @@
-VERSION = 1
+VERSION = 2
"""

multi_parsed = parse_diff(complex_diff)
print(f"Files parsed: {len(multi_parsed)}")
for f in multi_parsed:
    print(f"\nFile: {f['file']}")
    print(f"  Added lines  : {len(f['added_lines'])}")
    print(f"  Removed lines: {len(f['removed_lines'])}")

assert len(multi_parsed) == 3
assert multi_parsed[0]["file"] == "src/config.py"
assert multi_parsed[1]["file"] == "src/services/new_service.py"
assert multi_parsed[2]["file"] == "new_script.py"
print("\nTest 2 (Complex Multi-File Diff): PASSED")

Files parsed: 3

File: src/config.py
  Added lines  : 1
  Removed lines: 0

File: src/services/new_service.py
  Added lines  : 3
  Removed lines: 0

File: new_script.py
  Added lines  : 1
  Removed lines: 1

Test 2 (Complex Multi-File Diff): PASSED


### Test 3: `parse_diff` - Edge Cases
Verifies empty diffs, binary diffs, and diffs with `\ No newline at end of file`.

In [4]:
# Empty diff
assert parse_diff("") == []
assert parse_diff("   \n  ") == []

# Binary file
binary_diff = """diff --git a/assets/banner.png b/assets/banner.png
Binary files a/assets/banner.png and b/assets/banner.png differ
"""
bin_res = parse_diff(binary_diff)
assert len(bin_res) == 1
assert bin_res[0]["file"] == "assets/banner.png"
assert bin_res[0]["added_lines"] == []

print("Test 3 (Edge Cases): ALL PASSED")

Test 3 (Edge Cases): ALL PASSED


### Test 4: `comment_poster` - Validation & Error Handling
Verifies client creation and input validation on comments and check runs without requiring real credentials.

In [5]:
# 1. Test invalid pr_number
res1 = post_summary_comment("owner/repo", pr_number=-1, body="Test")
assert res1["success"] is False
print("Invalid PR # rejected :", res1["detail"])

# 2. Test empty body
res2 = post_summary_comment("owner/repo", pr_number=1, body="")
assert res2["success"] is False
print("Empty body rejected   :", res2["detail"])

# 3. Test invalid check run status
res3 = create_check_run("owner/repo", sha="abc", name="Review", status="unknown_status")
assert res3["success"] is False
print("Bad status rejected   :", res3["detail"])

print("\nTest 4 (Input Validation): ALL PASSED")

Invalid PR # rejected : pr_number must be a positive integer.
Empty body rejected   : Comment body must not be empty.
Bad status rejected   : Status 'unknown_status' is invalid. Must be one of: ['completed', 'in_progress', 'queued']

Test 4 (Input Validation): ALL PASSED


### Test 5: `post_summary_comment` (Mock Verification)
Simulates posting a summary comment to a PR using a mocked PyGithub Repository.

In [6]:
mock_repo = MagicMock()
mock_pr = MagicMock()
mock_comment = MagicMock()
mock_comment.id = 554433
mock_comment.html_url = "https://github.com/owner/repo/pull/42#issuecomment-554433"

mock_repo.get_pull.return_value = mock_pr
mock_pr.create_issue_comment.return_value = mock_comment

result = post_summary_comment(
    repo=mock_repo,
    pr_number=42,
    body="## AI Code Review Results\n**Verdict**: `request_changes`\n- Found 1 critical security finding.",
)

print("Summary Comment Result:")
print(json.dumps(result, indent=2))

assert result["success"] is True
assert result["comment_id"] == 554433
mock_pr.create_issue_comment.assert_called_once()
print("\nTest 5 (Summary Comment Mock): PASSED")

Summary Comment Result:
{
  "success": true,
  "comment_id": 554433,
  "url": "https://github.com/owner/repo/pull/42#issuecomment-554433"
}

Test 5 (Summary Comment Mock): PASSED


### Test 6: `post_inline_comment` (Mock Verification)
Simulates posting an inline discussion review comment attached to line 25 of `src/auth.py`.

In [7]:
mock_repo = MagicMock()
mock_pr = MagicMock()
mock_pr.head.sha = "head_sha_commit_789"
mock_inline = MagicMock()
mock_inline.id = 998877
mock_inline.html_url = "https://github.com/owner/repo/pull/42#discussion_r998877"

mock_repo.get_pull.return_value = mock_pr
mock_pr.create_review_comment.return_value = mock_inline

result = post_inline_comment(
    repo=mock_repo,
    pr_number=42,
    file="src/auth.py",
    line=25,
    body="⚠️ **SQL Injection**: Parameter `{user_id}` is directly interpolated. Use prepared statements.",
)

print("Inline Comment Result:")
print(json.dumps(result, indent=2))

assert result["success"] is True
assert result["comment_id"] == 998877
mock_pr.create_review_comment.assert_called_once_with(
    body="⚠️ **SQL Injection**: Parameter `{user_id}` is directly interpolated. Use prepared statements.",
    commit="head_sha_commit_789",
    path="src/auth.py",
    line=25,
    side="RIGHT",
)
print("\nTest 6 (Inline Comment Mock): PASSED")

Inline Comment Result:
{
  "success": true,
  "comment_id": 998877,
  "url": "https://github.com/owner/repo/pull/42#discussion_r998877"
}

Test 6 (Inline Comment Mock): PASSED


### Test 7: `create_check_run` with 50-Item Batching (Mock Verification)
GitHub limits Check Run annotations to 50 items per API call. This verifies that 70 annotations are automatically batched into two API calls (50 initial + 20 via `.edit()`).

In [ ]:
mock_repo = MagicMock()
mock_check = MagicMock()
mock_check.id = 112233
mock_check.html_url = "https://github.com/owner/repo/runs/112233"
mock_repo.create_check_run.return_value = mock_check

# Create 70 annotations
annotations_70 = [
    {
        "path": f"src/service_{i}.py",
        "start_line": i,
        "end_line": i,
        "annotation_level": "warning",
        "message": f"Warning on line {i}",
    }
    for i in range(1, 71)
]

result = create_check_run(
    repo=mock_repo,
    sha="commit_sha_12345",
    name="AI Code Reviewer Check",
    status="completed",
    conclusion="neutral",
    annotations=annotations_70,
)

print("Check Run Result:")
print(json.dumps(result, indent=2))

assert result["success"] is True
assert result["check_run_id"] == 112233

# Verify first batch of 50
first_batch = mock_repo.create_check_run.call_args[1]["output"]["annotations"]
print(f"First batch annotations  : {len(first_batch)} (Expected: 50)")
assert len(first_batch) == 50

# Verify second batch of 20 via .edit()
second_batch = mock_check.edit.call_args[1]["output"]["annotations"]
print(f"Second batch annotations : {len(second_batch)} (Expected: 20)")
assert len(second_batch) == 20

print("\nTest 7 (Check Run Batching): PASSED")

### Test 8: Live GitHub API Call (Optional)
If you set `GITHUB_TOKEN=your_token` in `agent-service/.env`, this cell will run against a real repository to fetch information.

In [ ]:
token = os.getenv("GITHUB_TOKEN")
if token and not token.startswith("your_"):
    try:
        client = get_github_client(token)
        user = client.get_user()
        print("Successfully connected to GitHub as:", user.login)
        print("GitHub API Rate Limit remaining:", client.get_rate_limit().core.remaining)
    except Exception as e:
        print("Failed to connect to GitHub:", e)
else:
    print("Notice: GITHUB_TOKEN not configured in .env.")
    print("To test against live GitHub, add to agent-service/.env:")
    print("  GITHUB_TOKEN=ghp_yourPersonalAccessTokenHere")